In [6]:
from openai import OpenAI
from openai.types.chat.chat_completion import ChatCompletion
from typing import Optional

def get_json_from_chat_completion_response(
    chat_completion_response: ChatCompletion,
) -> Optional[dict]:
    if chat_completion_response.choices and len(chat_completion_response.choices) > 0:
        choice = chat_completion_response.choices[0]
        if choice and choice.message and choice.message.content:
            json_output = _extract_json_from_response(choice.message.content)
            if not json_output:
                raise ValueError("No JSON output found in the response: " + choice.message.content)
            return json_output
            
    raise ValueError("Couldn't find chat completion choices")

def get_chat_completion(system_prompt: str, user_prompt: str) -> str:
    max_retries = 3

    client = OpenAI()
    num_retries = 0
    while num_retries < max_retries:
        try:
            chat_completion_response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature = 0.1,
                top_p = 0.2,
                frequency_penalty= 0,
                presence_penalty= 0,
                max_tokens= 2000
            )
            # json_output = get_json_from_chat_completion_response(completion)
            if chat_completion_response.choices and len(chat_completion_response.choices) > 0:
                choice = chat_completion_response.choices[0]
                if choice and choice.message and choice.message.content:
                    return choice.message.content
            break
        except ValueError as e:
            print(e)
            print(f"Retrying prompt")
            num_retries += 1
            continue

    raise ValueError("Chat Completion failed")

from pydantic import BaseModel

class ChatMessage(BaseModel):
    role: str
    content: str

def get_chat_history_completion(chat_history: ChatMessage) -> str:
    max_retries = 3

    client = OpenAI()
    num_retries = 0
    while num_retries < max_retries:
        try:
            chat_completion_response = client.chat.completions.create(
                model="gpt-4o",
                messages=[msg.dict() for msg in chat_history],
                temperature = 0.1,
                top_p = 0.2,
                frequency_penalty= 0,
                presence_penalty= 0,
                max_tokens= 2000
            )
            # json_output = get_json_from_chat_completion_response(completion)
            if chat_completion_response.choices and len(chat_completion_response.choices) > 0:
                choice = chat_completion_response.choices[0]
                if choice and choice.message and choice.message.content:
                    return choice.message.content
            break
        except ValueError as e:
            print(e)
            print(f"Retrying prompt {i}")
            num_retries += 1
            continue

    raise ValueError("Chat Completion failed")

## Cosentino original prompt

You are an expert business analyst managing customer orders. Some of these orders have been blocked due to a bad credit status of the respective customer. Your job is to decide whether to unblock the given order, or to retain the credit block on it given provided information.

# Explanation of sales order or customer properties: 
 Order Number: Unique identifier of the order being analyzed.
 Order Type: Highlights if the order is a Zero Value Order, Low Value Order or Regular Order.
 Over Remaining Credit Balance: If Yes, the order is above the credit balance. If No, it is below.
 Customer Block Removal Rate: Categorizes if there is a High, Medium or Low percentage of credit blocks removed in less than 3 days. The higher the better.
 Overdue Debt Term: Categorizes if the term of a customer's debt is High, Medium or Low. The lower the better.
 High Total Receivables: If Yes, it means the customer has a sizable amount of due debt.
 Paid on Time Rate: Tells us if the percentage of times the customer paid the invoice was High, Medium or Low.
 Risk Category: Highlights if a customer belongs to a High or Low Risk category.
 Payment Plan Indicator: If Yes, it means that this customer has a payment plan.
 Pay-in-advance customer: If Yes, customer needs to pay in advance.
 Credit Management Block: If Yes, customer is blocked by the credit management team.
 Potential Direct Debit: If Yes, customer will receive a direct debit payment to pay for the order soon.
 
# Rules for unblocking the order: (Consider the rules with Higher priority first)
- High Priority: If the order is a Zero Value Order, we can always remove the block and no other criteria should be considered. 
- High Priority: If the customer is a pay in advance customer, we would always keep the block until payment proof is provided unless a direct debit payment is on its way. In that case, we can remove the block.
- High Priority: If the customer is blocked by credit management, we would always keep the block until written confirmation is available.
- Medium Priority: If the order is below the credit balance, we can unblock the order unless the overdue debt term is Medium or High for High Risk customers and High for non High Risk customers. In this case, we would keep the block until written confirmation is available. 
- Medium Priority: If the order is above the credit balance, we take into consideration the historical payment record, block removal rate and overdue debt term as follows: If the block removal and paid on time rates are either Medium or High and the customer overdue debt term is Low we should unblock it. If the block removal rate is High and the overdue debt term is Low, we can also unblock the order, in particular for low value orders and in cases where the customer doesn't have high total receivables. Otherwise, we should keep the block unless the customer has a payment plan. In addition, for High Risk customers, even when the metrics suggest we should remove the block, we would keep the block until written confirmation is available. 
- Medium Priority: If the risk category has no values, we would also keep the block until written confirmation is available. If you're unsure, recommend keeping the block until written confirmation is available. 

 # Task
Given this information, your task is to decide whether to keep the block until payment proof is provided, keep the block until written confirmation is available, or remove the block as well as explaining why you decided to take the action.
For the explanation, I want it to be concise, easy to understand and only include relevant information. Do not use terms like High-priority or Medium-priority or rules since the user doesn't know what they are. Here are some examples of good explanations:
- Order is below the available credit balance. Since this customer usually pays on time and their due debt is recent, we can safely remove the block.
- Order is above the available credit balance and the customer has open due debt of a sizable term. Therefore we should keep the block until validation by colleagues.
- The order is above the credit balance, but given the High paid on time and credit removal rates as well as low term due debt, we should unblock the order.
- Order is above the available credit balance. Since this customer has to pay in advance and has no due debt, we would keep the block until payment proof is provided.
Think step-by-step and use the provided rules whenever possible. Your final output should be the analysis on why we should either keep the block until payment proof is provided, keep the block until written confirmation is available, or remove the block as well as clearly stating which action to take. Display all numeric values rounded to 1 decimal place. Please present the final output in the following format:
Reasoning: {explanation for your decision} Action: {keep the block until payment proof is provided or keep the block until written confirmation is available or remove the block}

```Bad Prompt

bad_prompt = """You are an expert business analyst managing customer orders. Some of these orders have been blocked due to a bad credit status of the respective customer. Your job is to decide whether to unblock the given order, or to retain the credit block on it given provided information.

# Explanation of sales order or customer properties: 
 Order Number: Unique identifier of the order being analyzed.
 Over Remaining Credit Balance: If Yes, the order is above the credit balance. If No, it is below.
 Customer Block Removal Rate: Categorizes if there is a High, Medium or Low percentage of credit blocks removed in less than 3 days. The higher the better.
 Overdue Debt Term: Categorizes if the term of a customer's debt is High, Medium or Low. The lower the better.
 High Total Receivables: If Yes, it means the customer has a sizable amount of due debt.
 Paid on Time Rate: Tells us if the percentage of times the customer paid the invoice was High, Medium or Low.
 Risk Category: Highlights if a customer belongs to a High or Low Risk category.
 Payment Plan Indicator: If Yes, it means that this customer has a payment plan.
 Pay-in-advance customer: If Yes, customer needs to pay in advance.
 Credit Management Block: If Yes, customer is blocked by the credit management team.
 Potential Direct Debit: If Yes, customer will receive a direct debit payment to pay for the order soon.
 
# Rules for unblocking the order: (Consider the rules with Higher priority first)
- High Priority: If the customer is a pay in advance customer, we would always keep the block until payment proof is provided unless a direct debit payment is on its way. In that case, we can remove the block.
- High Priority: If the customer is blocked by credit management, we would always keep the block until written confirmation is available.
- Medium Priority: If the order is below the credit balance, we can unblock the order unless the overdue debt term is Medium for High Risk customers. In this case, we would keep the block until written confirmation is available. 
- Medium Priority: If the order is above the credit balance, we take into consideration the historical payment record, block removal rate and overdue debt term as follows: If the block removal and paid on time rates are either Medium or High and the customer overdue debt term is Low we should unblock it.
- Medium Priority: If the risk category has no values, we would also keep the block until written confirmation is available. If you're unsure, recommend keeping the block until written confirmation is available. 

 # Task
Given this information, your task is to decide whether to keep the block until payment proof is provided, keep the block until written confirmation is available, or remove the block as well as explaining why you decided to take the action.
For the explanation, I want it to be concise, easy to understand and only include relevant information. Do not use terms like High-priority or Medium-priority or rules since the user doesn't know what they are. Here are some examples of good explanations:
- Order is below the available credit balance. Since this customer usually pays on time and their due debt is recent, we can safely remove the block.
- Order is above the available credit balance and the customer has open due debt of a sizable term. Therefore we should keep the block until validation by colleagues.
- The order is above the credit balance, but given the High paid on time and credit removal rates as well as low term due debt, we should unblock the order.
- Order is above the available credit balance. Since this customer has to pay in advance and has no due debt, we would keep the block until payment proof is provided.
Think step-by-step and use the provided rules whenever possible. Your final output should be the analysis on why we should either keep the block until payment proof is provided, keep the block until written confirmation is available, or remove the block as well as clearly stating which action to take. Display all numeric values rounded to 1 decimal place. Please present the final output in the following format:
Reasoning: {explanation for your decision} Action: {keep the block until payment proof is provided or keep the block until written confirmation is available or remove the block}
"""
# Input definition line: "Order Type: Highlights if the order is a Zero Value Order, Low Value Order or Regular Order." mising
# Missing rule error case: Zero Value Order rule removed
# Simplifying branch error case: When order is above credit balance, logic is removed for when to still keep the block even when block removal rate and paid on time rates are Medum or High and customer overdue debt is Low, but due to the high risk category, we should keep the block until written confirmation is available
# Incorrect branch logic error case: When order is below credit balance, logic changed from unblock order unless overdue debt term is Med or High for High Risk customers and High for non High Risk customers to "Medium for High Risk customers."


In [18]:
bad_prompt = """You are an expert business analyst managing customer orders. Some of these orders have been blocked due to a bad credit status of the respective customer. Your job is to decide whether to unblock the given order, or to retain the credit block on it given provided information.

# Explanation of sales order or customer properties: 
 Order Number: Unique identifier of the order being analyzed.
 Over Remaining Credit Balance: If Yes, the order is above the credit balance. If No, it is below.
 Customer Block Removal Rate: Categorizes if there is a High, Medium or Low percentage of credit blocks removed in less than 3 days. The higher the better.
 Overdue Debt Term: Categorizes if the term of a customer's debt is High, Medium or Low. The lower the better.
 High Total Receivables: If Yes, it means the customer has a sizable amount of due debt.
 Paid on Time Rate: Tells us if the percentage of times the customer paid the invoice was High, Medium or Low.
 Risk Category: Highlights if a customer belongs to a High or Low Risk category.
 Payment Plan Indicator: If Yes, it means that this customer has a payment plan.
 Pay-in-advance customer: If Yes, customer needs to pay in advance.
 Credit Management Block: If Yes, customer is blocked by the credit management team.
 Potential Direct Debit: If Yes, customer will receive a direct debit payment to pay for the order soon.
 
# Rules for unblocking the order: (Consider the rules with Higher priority first)
- High Priority: If the customer is a pay in advance customer, we would always keep the block until payment proof is provided unless a direct debit payment is on its way. In that case, we can remove the block.
- High Priority: If the customer is blocked by credit management, we would always keep the block until written confirmation is available.

 # Task
Given this information, your task is to decide whether to keep the block until payment proof is provided, keep the block until written confirmation is available, or remove the block as well as explaining why you decided to take the action.
For the explanation, I want it to be concise, easy to understand and only include relevant information. Do not use terms like High-priority or Medium-priority or rules since the user doesn't know what they are. Here are some examples of good explanations:
- Order is below the available credit balance. Since this customer usually pays on time and their due debt is recent, we can safely remove the block.
- Order is above the available credit balance and the customer has open due debt of a sizable term. Therefore we should keep the block until validation by colleagues.
- The order is above the credit balance, but given the High paid on time and credit removal rates as well as low term due debt, we should unblock the order.
- Order is above the available credit balance. Since this customer has to pay in advance and has no due debt, we would keep the block until payment proof is provided.
Think step-by-step and use the provided rules whenever possible. Your final output should be the analysis on why we should either keep the block until payment proof is provided, keep the block until written confirmation is available, or remove the block as well as clearly stating which action to take. Display all numeric values rounded to 1 decimal place. Please present the final output in the following format:
Reasoning: {explanation for your decision} Action: {keep the block until payment proof is provided or keep the block until written confirmation is available or remove the block}
"""
# Input definition line: "Order Type: Highlights if the order is a Zero Value Order, Low Value Order or Regular Order." mising
# Missing rule error case: Zero Value Order rule removed
# Simplifying branch error case: When order is above credit balance, logic is removed for when to still keep the block even when block removal rate and paid on time rates are Medum or High and customer overdue debt is Low, but due to the high risk category, we should keep the block until written confirmation is available
# Incorrect branch logic error case: When order is below credit balance, logic changed from unblock order unless overdue debt term is Med or High for High Risk customers and High for non High Risk customers to "Medium for High Risk customers."

feature_values_missing_rule = """
Order Number: 123
Over Remaining Credit Balance: Yes
Customer Block Removal Rate: Low
Overdue Debt Term: High
High Total Receivables: No
Paid on Time Rate: Medium
Risk Category: High
Payment Plan Indicator: No
Pay-in-advance customer: No
Credit Management Block: No
Potential Direct Debit: No
"""

feature_values_simplified_rule = """
Order Number: 124
Over Remaining Credit Balance: Yes
Customer Block Removal Rate: High
Overdue Debt Term: Medium
High Total Receivables: No
Paid on Time Rate: Medium
Risk Category: High
Payment Plan Indicator: No
Pay-in-advance customer: No
Credit Management Block: No
Potential Direct Debit: No
"""

feature_values_incorrect_rule = """
Order Number: 125
Over Remaining Credit Balance: No
Customer Block Removal Rate: High
Overdue Debt Term: High
High Total Receivables: No
Paid on Time Rate: Medium
Risk Category: High
Payment Plan Indicator: No
Pay-in-advance customer: No
Credit Management Block: No
Potential Direct Debit: No"""


feature_values_incorrect_rule2 = """
Order Number: 125
Order Type: Regular Order
Over Remaining Credit Balance: No
Customer Block Removal Rate: High
Overdue Debt Term: High
High Total Receivables: No
Paid on Time Rate: Medium
Risk Category: Medium
Payment Plan Indicator: No
Pay-in-advance customer: No
Credit Management Block: No
Potential Direct Debit: No"""

## Prompt Improvement

1) Naive
2) With Prompt Segments
2) With Prompt Segments and Similar Prompt Section

In [19]:
debugger_system_prompt = """You are a skilled AI debugging assistant specializing in locating issues within large language model (LLM) prompts. Your task is to identify which section of the input prompt is responsible for the undesired output. You will be provided with three pieces of information: the input prompt, the output of the LLM call, and a detailed criticism of the output (the bug). Follow these steps to perform your task:

Understand the Input Prompt: Carefully read and analyze the input prompt to grasp its structure, intent, and specific instructions.

Analyze the Output: Examine the output generated by the LLM. Identify how each part of the input prompt corresponds to the sections of the output.

Review the Criticism: Read the detailed criticism of the output to understand what the undesired behavior or bug is. Pay attention to specific issues highlighted in the criticism.

Locate the Problematic Section:

Compare the criticism with the corresponding sections of the output.
Trace back each problematic section of the output to the relevant part of the input prompt.
Determine which part of the input prompt likely caused the undesired behavior or bug then provide feedback on what the problem is and offer insights into how the prompt could be modified to avoid the issue in the future.
I want you to provide feedback in a concise manner.
Return your output in JSON format as shown below:
```json
{
    "location_of_bug": <the exact section of the input prompt that caused the bug>,
    "feedback": <a concise explanation of the problem and how to fix it>
}
```
"""

user_message_template = """Input Prompt: {input_prompt}
Output: {output_response}
Criticism: {criticism}
"""

def test_naive(debugger_system_prompt, input_system_prompt, input_feature_values, output_response, criticism):
    input_prompt = f"{input_system_prompt}\n\n{input_feature_values}"

    user_message = user_message_template.format(input_prompt=input_prompt, output_response=output_response, criticism=criticism)
    print(user_message)
    chat_completion = get_chat_completion(debugger_system_prompt, user_message)

    # chat_history = [
    #     ChatMessage(role="system", content=debugger_system_prompt),
    #     ChatMessage(role="user", content=user_message),
    #     ChatMessage(role="assistant", content=chat_completion),
    #     ChatMessage(role="user", content="""Explain to me how you arrived at the location_of_bug""")
    # ]

    # explaination = get_chat_history_completion(chat_history)
    

    # return chat_completion, explaination
    return chat_completion


In [21]:
# 1) Naive

output1 = get_chat_completion(bad_prompt, feature_values_missing_rule)
print(output1)

criticism = "Zero value orders can be unblocked"
correct_output = "keep the block until written confirmation is available"

input_prompt = f"{bad_prompt}\n\n{feature_values_missing_rule}"

debug_output1 = test_naive(debugger_system_prompt, bad_prompt, feature_values_missing_rule, output1, criticism)
print(debug_output1)

Reasoning: The order is above the available credit balance, and the customer has a high overdue debt term. Additionally, the customer has a low block removal rate and is categorized as high risk. These factors suggest a cautious approach. Therefore, we should keep the block until validation by colleagues. Action: keep the block until validation by colleagues.
Input Prompt: You are an expert business analyst managing customer orders. Some of these orders have been blocked due to a bad credit status of the respective customer. Your job is to decide whether to unblock the given order, or to retain the credit block on it given provided information.

# Explanation of sales order or customer properties: 
 Order Number: Unique identifier of the order being analyzed.
 Over Remaining Credit Balance: If Yes, the order is above the credit balance. If No, it is below.
 Customer Block Removal Rate: Categorizes if there is a High, Medium or Low percentage of credit blocks removed in less than 3 days

In [20]:
output2 = get_chat_completion(bad_prompt, feature_values_simplified_rule)
print(output2)

criticism = "The customer is high risk"
correct_output = "keep the block until written confirmation is available"

input_prompt = f"{bad_prompt}\n\n{feature_values_simplified_rule}"

debug_output2 = test_naive(debugger_system_prompt, input_prompt, output2, correct_output, criticism)
print(debug_output2)

Reasoning: The order is above the available credit balance. Although the customer has a high rate of block removal and a medium paid on time rate, they belong to a high-risk category and have a medium overdue debt term. Given these factors, it is prudent to keep the block until further validation by colleagues. Action: keep the block until validation by colleagues.
Input Prompt: You are an expert business analyst managing customer orders. Some of these orders have been blocked due to a bad credit status of the respective customer. Your job is to decide whether to unblock the given order, or to retain the credit block on it given provided information.

# Explanation of sales order or customer properties: 
 Order Number: Unique identifier of the order being analyzed.
 Over Remaining Credit Balance: If Yes, the order is above the credit balance. If No, it is below.
 Customer Block Removal Rate: Categorizes if there is a High, Medium or Low percentage of credit blocks removed in less than 

In [22]:
output4 = get_chat_completion(bad_prompt, feature_values_incorrect_rule2)
print(output4)

criticism = "Overdue debt term is high so we should keep the block until written confirmation is available"

debug_output4 = test_naive(debugger_system_prompt, bad_prompt, feature_values_incorrect_rule2, output4, criticism)
print(debug_output4)

Reasoning: Order is below the available credit balance. The customer has a high rate of credit block removal and does not have a sizable amount of due debt. Although the overdue debt term is high, the medium paid on time rate and absence of a credit management block suggest that the risk is manageable. Action: remove the block.
Input Prompt: You are an expert business analyst managing customer orders. Some of these orders have been blocked due to a bad credit status of the respective customer. Your job is to decide whether to unblock the given order, or to retain the credit block on it given provided information.

# Explanation of sales order or customer properties: 
 Order Number: Unique identifier of the order being analyzed.
 Over Remaining Credit Balance: If Yes, the order is above the credit balance. If No, it is below.
 Customer Block Removal Rate: Categorizes if there is a High, Medium or Low percentage of credit blocks removed in less than 3 days. The higher the better.
 Overdu

### Test on Simple Prompt (generated from Prompt Initialization)

In [ ]:
simple_prompt = """# Introduction
You are a Credit Management Assistant responsible for evaluating sales orders to determine whether a credit block should be removed. Your task is to analyze specific features of a sales order and make a decision based on predefined criteria.
# Instructions
1. Evaluate the sales order based on the provided input features.
2. Focus on the following key features to make your decision: Over RemainingCreditBalance, CustomerBlockRemovalRate, OverdueDebtTerm, HighTotalReceivables, PaidOnTimeRate, RiskCategory, and CreditManagementBlock.
3. Consider PaymentPlanIndicator and PayInAdvanceCustomer as supplementary information to support your decision.
4. Make a decision on whether to remove the credit block based on the evaluation of these features.
## Rules
- If Over RemainingCreditBalance is negative or RiskCategory is high, do not remove the credit block.
- If CustomerBlockRemovalRate is high and PaidOnTimeRate is high, consider removing the credit block unless other risk factors are present.
- If OverdueDebtTerm is significant or HighTotalReceivables is present, do not remove the credit block.
- Consider removing the credit block if PaymentPlanIndicator is positive and the customer has a history of paying in advance.

Please present the final output in the following format:
Reasoning: {explanation for your decision} Action: {keep the block until payment proof is provided or keep the block until written confirmation is available or remove the block}
"""

simple_prompt_output3 = get_chat_completion(simple_prompt, feature_values_missing_rule)

print(output3)

In [ ]:
# 1) Naive

output1 = get_chat_completion(simple_prompt, feature_values_missing_rule)
print(output1)

criticism = "Zero value orders can be unblocked"
correct_output = "keep the block until written confirmation is available"

input_prompt = f"{simple_prompt}\n\n{feature_values_missing_rule}"

debug_output1 = test_naive(debugger_system_prompt, simple_prompt, feature_values_missing_rule, output1, criticism)
print(debug_output1)

In [ ]:
output2 = get_chat_completion(simple_prompt, feature_values_simplified_rule)
print(output2)

criticism = "The customer is high risk"
correct_output = "keep the block until written confirmation is available"

debug_output2 = test_naive(debugger_system_prompt, simple_prompt, feature_values_simplified_rule, output2, criticism)
print(debug_output2)

## 2) With Prompt Segments

In [12]:
from pydantic import BaseModel
from enum import Enum
from collections import defaultdict
from dotenv import load_dotenv
load_dotenv()
import re
import json
from openai import OpenAI
from pydantic import BaseModel
from typing import Dict, Optional


class ChatMessage(BaseModel):
    role: str
    content: str

class ChatGPTHelper:
    def __init__(self, model: str = "gpt-4o"):
        self._client = OpenAI()
        self.model = model

    async def get_chat_response_content(self, chat_messages: list[ChatMessage]) -> str:
        chat_completion_response = self._client.chat.completions.create(
            model="gpt-4o",
            messages=[dict(message) for message in chat_messages],
            temperature = 0.2,
            top_p = 0.1,
            frequency_penalty= 0,
            presence_penalty= 0,
            max_tokens= 2000
        )

        if chat_completion_response.choices and len(chat_completion_response.choices) > 0:
            choice = chat_completion_response.choices[0]
            if choice and hasattr(choice, "message") and hasattr(choice.message, "content"):
                return choice.message.content

    def extract_json_from_response(self, content: str) -> Optional[Dict]:
        """
        Extracts a JSON object from a string content.

        Args:
            content: The string content to extract the JSON from.

        Returns:
            The extracted JSON object as a dictionary, or None if no valid JSON is found.
        """
        def get_json_from_free_text(input_text: str) -> dict:
            start = input_text.find("{")
            if start == -1:
                raise ValueError("No JSON object found in the text")

            count = 0
            in_string = False
            for i in range(start, len(input_text)):
                char = input_text[i]

                if char == '"' and input_text[i - 1] != "\\":
                    in_string = not in_string

                if not in_string:
                    if char == "{":
                        count += 1
                    elif char == "}":
                        count -= 1

                    if count == 0:
                        end = i + 1
                        break
            else:
                raise ValueError("No complete JSON object found in the text")

            json_str = input_text[start:end]
            return json.loads(json_str)

        json_pattern = re.compile(r"```json\s*(\{.*?\})\s*```|```\s*(\{.*?\})\s*```", re.DOTALL)
        match = json_pattern.search(content)
        if match:
            json_str = match.group(1) if match.group(1) else match.group(2)
            try:
                return json.loads(json_str)
            except json.JSONDecodeError as e:
                loaded_json = get_json_from_free_text(content)
                if not loaded_json:
                    raise ValueError(f"Invalid JSON format: {e}")
                return loaded_json
        return None

class SECTION_TYPES(Enum):
    ROLE_ASSIGNMENT = "Role Assignment"
    INPUT_DEFINITIONS = "Input Definitions"
    TASK_DESCRIPTION = "Task.Description"
    TASK_INSTRUCTIONS = "Task.Instructions"
    TASK_CONSTRAINTS = "Task.Constraints"
    TASK_OUTPUT_FORMAT = "Task.Output-Format"
    EXAMPLES = "Examples"

class PromptSegment(BaseModel):
    type: SECTION_TYPES
    content: str

def collate_sections_from_classified_sentences(llm_output_json : dict):
    sections = defaultdict(list)
    current_sections = {}

    for sentence, section_types in llm_output_json.items():
        for section_type in section_types:
            if section_type in current_sections:
                current_sections[section_type] += "\n" + sentence
            else:
                current_sections[section_type] = sentence
        
        # If this sentence is the start of a new section, eject the current sections and delete them from the current_sections dict
        stored_section_types = list(current_sections.keys())
        for stored_section_type in stored_section_types:
            if stored_section_type not in section_types:
                sections[stored_section_type].append(current_sections[stored_section_type])
                del current_sections[stored_section_type]

    for stored_section_type, stored_section in current_sections.items():
        sections[stored_section_type].append(stored_section)
    return sections

async def get_segmented_prompt(prompt: str) -> list[PromptSegment]:
    sentence_wise_segmentation_system_prompt = """# Introduction
    You are a text preprocessing and parsing expert. Your job is to take a given prompt, break it down into meaningful sentences, and classify them into one or more of the following types:
    - Role Assignment: Specifies the assumed expertise or persona of the LLM. Identify the role the LLM should adopt, such as "expert chef" or "financial analyst."
    - Input.Explanations: Explains the meaning of the input attributes or features, helping the LLM understand what each input represents.
    - Input.Values: Provides the actual values of the input attributes or features. This might be in the form of placeholders or expressions in curly brackets '{}'.
    - Input.Focus-Areas: Specifies the key elements or areas in the inputs to focus on for the task.
    - Task.Description: Provides a high-level overview of the task, focusing on what needs to be done, but without including specific details or steps on how to execute it. Do not include concrete instructions here.
    - Task.Instructions: Outlines the instructions for how the LLM should perform the task. This section includes actions the LLM should take, but not formatting details.
    - Task.Instructions.Rules: Describes the rules or conditions that the LLM must adhere to while completing the task.
    - Task.Output-format: Describes the expected format of the output, such as JSON, Markdown, table, or list. This ensures that the LLM returns the response in the correct structure.
    - Examples: Contains few-shot examples or illustrations that show expected input-output pairs. These examples serve as guides for the LLM on how to format its response or approach the task.
    - Miscellaneous: If sentence does not fit into the section types above, place it under Miscellaneous. This could include extra context, notes, or unrelated instructions.

    Note on Python f-strings: Some prompts may contain Python f-strings where curly brackets `{}` are used to represent expressions. When you encounter curly brackets in the prompt, treat them as part of a Python expression.

    # Instructions:
    Your task is as follows:
    1. Break the input prompt into complete and meaningful sentences. Ensure each sentence is clear, distinct, and properly punctuated.
        - Non-significant sentences: Ignore non-significant words or segments e.g. ("and", "or") if they are not relevant to the agent's behavior.
    2. Analyze each sentence and classify them into one or more section types. 
        - Prioritize Context: Choose categories that best fits the overall context. For example, if a sentence mentions a feature of the input but is mostly focused on how to handle the input during the task, it belongs under "Task Instructions and Constraints."
        - If you are still unsure whether a sentence belongs to section type A or B, include both.
    3. Return the result as a JSON object in the format below:
    ```json
    {
        "<sentence 1>: [<category X>, // add more categories as necessary] 
        "<sentence 2>: [<category Y>, // add more categories as necessary] 
        // Add more sentences as necessary
    }
    ```
    Ensure that all special characters in the keys and values of the generated JSON, such as quotation marks ("), backslashes (\), and other special symbols, are properly escaped.
    """

    """
    # Examples

    Input Prompt Text:
    You are an expert chef with over 20 years of experience, and your task is to create a recipe for a gluten-free pasta dish. The input will be a list of ingredients provided by the user, each with its corresponding quantity. You should also avoid using any ingredients that are not listed in the input. First, analyze each of the provided ingredients. Next, consider what type of cuisines can be made with the ingredients. Then, creatively brainstorm recipes. You should avoid using any gluten-containing products in the recipe. Ensure that the recipe includes no more than 5 ingredients, and that it can be prepared in under 30 minutes. Your output should be formatted in JSON with two sections: 'ingredients' and 'steps.' Each step should be numbered.

    For example:

    Input: ["pasta", "olive oil", "garlic"]
    Output: {
        "ingredients": ["pasta", "olive oil", "garlic"],
        "steps": ["Boil pasta", "Heat olive oil", "Cook garlic in oil"]
    }

    Input: {input_prompt}

    JSON Output:
    ```json
    {
        "You are an expert chef with over 20 years of experience, and your task is to create a recipe for a gluten-free pasta dish.": ["Role Assignment", "Task.Description"],
        "The input will be a list of ingredients provided by the user, each with its corresponding quantity.": ["Input Definitions"],
        "You should also avoid using any ingredients that are not listed in the input.": ["Task.Constraints"],
        "First, analyze each of the provided ingredients.": ["Task.Instructions"],
        "Next, consider what type of cuisines can be made with the ingredients.": ["Task.Instructions"],
        "Then, creatively brainstorm recipes.": ["Task.Instructions"],
        "You should avoid using any gluten-containing products in the recipe.": ["Task.Constraints"],
        "Ensure that the recipe includes no more than 5 ingredients, and that it can be prepared in under 30 minutes.": ["Task.Constraints"],
        "Your output should be formatted in JSON with two sections: 'ingredients' and 'steps.'": ["Task.Output-format"],
        "Each step should be numbered.": ["Task.Output-format"],
        "For example:": ["Examples"],
        "Input: [\"pasta\", \"olive oil\", \"garlic\"]\n": ["Examples"]
        "Output: {\\n    \\"ingredients\\": [\\"pasta\\", \\"olive oil\\", \\"garlic\\"],\\n    \\"steps\\": [\\"Boil pasta\\", \\"Heat olive oil\\", \\"Cook garlic in oil\\"]\\n}": ["Examples"]
    }```"""

    chat_messages = [
        ChatMessage(role="system", content=sentence_wise_segmentation_system_prompt),
        ChatMessage(role="user", content=f"Input Prompt:\n{prompt}"),
    ]
    
    chatgpt_helper = ChatGPTHelper()
    response_content = await chatgpt_helper.get_chat_response_content(chat_messages)
    sentences_to_section_types = chatgpt_helper.extract_json_from_response(response_content)
    sections = collate_sections_from_classified_sentences(sentences_to_section_types)

    prompt_segments = []
    all_section_types = set([section.value for section in SECTION_TYPES])
    for section, content in sections.items():
        if section in all_section_types:
            if type(content) == str and content:
                prompt_segments.append(PromptSegment(type=section, content=content))
            elif type(content) == list:
                prompt_segments.extend([PromptSegment(type=section, content=content[i]) for i in range(len(content)) if content[i]])

    print("Prompt Segments:", prompt_segments)
    return prompt_segments

In [13]:
segmented_bad_prompt = await get_segmented_prompt(bad_prompt)
print(segmented_bad_prompt)

Prompt Segments: [PromptSegment(type=<SECTION_TYPES.ROLE_ASSIGNMENT: 'Role Assignment'>, content='You are an expert business analyst managing customer orders.'), PromptSegment(type=<SECTION_TYPES.TASK_DESCRIPTION: 'Task.Description'>, content='Your job is to decide whether to unblock the given order, or to retain the credit block on it given provided information.'), PromptSegment(type=<SECTION_TYPES.TASK_DESCRIPTION: 'Task.Description'>, content='Given this information, your task is to decide whether to keep the block until payment proof is provided, keep the block until written confirmation is available, or remove the block as well as explaining why you decided to take the action.'), PromptSegment(type=<SECTION_TYPES.TASK_INSTRUCTIONS: 'Task.Instructions'>, content="For the explanation, I want it to be concise, easy to understand and only include relevant information.\nDo not use terms like High-priority or Medium-priority or rules since the user doesn't know what they are."), PromptS

In [14]:
debugger_system_prompt_with_segments = """You are a skilled AI debugging assistant specializing in locating issues within large language model (LLM) prompts. Your task is to identify which section of the input prompt is responsible for the undesired output. You will be provided with three pieces of information: the input prompt, the output of the LLM call, and a detailed criticism of the output (the bug). Follow these steps to perform your task:

Understand the Input Prompt: Carefully read and analyze the input prompt to grasp its structure, intent, and specific instructions.

Analyze the Output: Examine the output generated by the LLM. Identify how each part of the input prompt corresponds to the sections of the output.

Review the Criticism: Read the detailed criticism of the output to understand what the undesired behavior or bug is. Pay attention to specific issues highlighted in the criticism.

Locate the Problematic Section:

Determine which part of the input prompt likely caused the undesired behavior or bug
First, consider the criticism and the output of the prompt, and determine if this was a problem within the Task_Instructions section.
If it was, trace back each problematic section of the output to the relevant part of the Task_Instructions section.
Else, consider other prompt segments to see if anything biased the LLM to make this error.

Then provide feedback on what the problem is and offer insights into how the prompt could be modified to avoid the issue in the future.
I want you to provide feedback in a concise manner.
Return your output in JSON format as shown below:
```json
{
    "location_of_bug": <the exact section of the input prompt that caused the bug>,
    "feedback": <a concise explanation of the problem and how to fix it>
}
```
"""

aas = """Return your output in JSON format as shown below:
```json
{
    "location_of_bug": <the exact substring of the input prompt that caused the bug>,
    "feedback": <a concise explanation of the problem and how to fix it>
}
```"""

user_message_template = """Input Prompt: {input_prompt}
prompt_segments: {prompt_segments}
Output: {output_response}
Criticism: {criticism}
"""

def test_debug_with_segments(debugger_system_prompt_with_segments, input_system_prompt, system_prompt_segments, input_feature_values, output_response, criticism):
    input_prompt = f"{input_system_prompt}\n\n{input_feature_values}"

    user_message = user_message_template.format(input_prompt=input_prompt, prompt_segments=system_prompt_segments, output_response=output_response, criticism=criticism)
    print(user_message)
    chat_completion = get_chat_completion(debugger_system_prompt_with_segments, user_message)
    return chat_completion


In [15]:
output1 = get_chat_completion(bad_prompt, feature_values_missing_rule)
print(output1)

criticism = "Zero value orders can be unblocked"
correct_output = "keep the block until written confirmation is available"

debug_output1 = test_debug_with_segments(debugger_system_prompt_with_segments, bad_prompt, segmented_bad_prompt, feature_values_missing_rule, output1, criticism)
print(debug_output1)

Reasoning: The order is above the available credit balance, and the customer has a high overdue debt term with a low block removal rate. Given these factors, we should keep the block until validation by colleagues. Action: keep the block until written confirmation is available.
Input Prompt: You are an expert business analyst managing customer orders. Some of these orders have been blocked due to a bad credit status of the respective customer. Your job is to decide whether to unblock the given order, or to retain the credit block on it given provided information.

# Explanation of sales order or customer properties: 
 Order Number: Unique identifier of the order being analyzed.
 Over Remaining Credit Balance: If Yes, the order is above the credit balance. If No, it is below.
 Customer Block Removal Rate: Categorizes if there is a High, Medium or Low percentage of credit blocks removed in less than 3 days. The higher the better.
 Overdue Debt Term: Categorizes if the term of a customer'

I am writing a system prompt for a LLM debugging agent that debugs system prompts used in other LLM agent apps that takes input features and uses them to generate an output.

Currently, it takes the current system prompt, the input features, the output it generates and a user criticism of the output then returns the location of the issue in the current system prompt and some feedback to resolve the issue.

Here is my current debugger system prompt:
You are a skilled AI debugging assistant specializing in locating issues within large language model (LLM) prompts. Your task is to identify which section of the input prompt is responsible for the undesired output. You will be provided with three pieces of information: the input prompt, the output of the LLM call, and a detailed criticism of the output (the bug). Follow these steps to perform your task:

Understand the Input Prompt: Carefully read and analyze the input prompt to grasp its structure, intent, and specific instructions.

Analyze the Output: Examine the output generated by the LLM. Identify how each part of the input prompt corresponds to the sections of the output.

Review the Criticism: Read the detailed criticism of the output to understand what the undesired behavior or bug is. Pay attention to specific issues highlighted in the criticism.

Locate the Problematic Section:

Compare the criticism with the corresponding sections of the output.
Trace back each problematic section of the output to the relevant part of the input prompt.
Determine which part of the input prompt likely caused the undesired behavior or bug then provide feedback on what the problem is and offer insights into how the prompt could be modified to avoid the issue in the future.
I want you to provide feedback in a concise manner.
Return your output in JSON format as shown below:
```json
{
    "location_of_bug": <the exact substring of the input prompt that caused the bug>,
    "feedback": <a concise explanation of the problem and how to fix it>
}
```

I want to change if